In [50]:
import ccxt
import pandas as pd
import time
import ta
from dotenv import load_dotenv

In [53]:
# Parámetros de trading
load_dotenv()
symbol = 'BTC/USDT'
timeframe = '1m'
risk_per_trade = 0.5  # Riesgo máximo por operación (50%)
entry_threshold = 0.001  # Umbral para entrar al mercado (0.1%)
position_min = 0.00001  # Tamaño mínimo de posición permitido
commission = 0.001  # Comisión estándar de Binance (0.1%)
minimum_trade_value = 5  # Valor mínimo permitido por operación en USD

# Configuración de simulación / trading real
simulation = True  # Cambia a False para trading real

exchange = ccxt.binance({
    'apiKey': os.getenv('API_KEY'),
    'secret': os.getenv('API_SECRET'),
    'enableRateLimit': True
})

capital_inicial = 10  # Capital inicial en USD
capital_actual = capital_inicial
trades = 0
comisiones_totales = 0

In [54]:
def get_balance():
    if simulation:
        return capital_actual
    else:
        balance = exchange.fetch_balance()
        return balance['total']['USDT']

def get_data():
    bars = exchange.fetch_ohlcv(symbol, timeframe, limit=100)
    df = pd.DataFrame(bars, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    return df

def generate_signals(df):
    df['rsi'] = ta.momentum.RSIIndicator(close=df['close'], window=14).rsi()
    df['sma_50'] = ta.trend.SMAIndicator(close=df['close'], window=50).sma_indicator()
    df['sma_200'] = ta.trend.SMAIndicator(close=df['close'], window=200).sma_indicator()
    df.dropna(inplace=True)

    last_row = df.iloc[-1]
    signal = None

    if last_row['rsi'] < 30 and last_row['sma_50'] > last_row['sma_200']:
        signal = 'buy'
    elif last_row['rsi'] > 70 and last_row['sma_50'] < last_row['sma_200']:
        signal = 'sell'

    return signal

def place_order(signal):
    global capital_actual, trades, comisiones_totales

    balance = get_balance()
    price = exchange.fetch_ticker(symbol)['last']
    amount = (balance * risk_per_trade) / price
    trade_value = amount * price

    if amount < position_min or trade_value < minimum_trade_value:
        print('El tamaño calculado es muy pequeño para realizar la operación o su valor es menor a 5 USD.')
        return

    amount -= amount * commission
    comision_actual = amount * commission
    comisiones_totales += comision_actual

    if simulation:
        if signal == 'buy':
            capital_actual -= trade_value + comision_actual
        elif signal == 'sell':
            capital_actual += trade_value - comision_actual
    else:
        if signal == 'buy':
            order = exchange.create_market_buy_order(symbol, amount)
            print(f'Orden de compra ejecutada: {order}')
        elif signal == 'sell':
            order = exchange.create_market_sell_order(symbol, amount)
            print(f'Orden de venta ejecutada: {order}')

    trades += 1

def run_bot():
    global capital_actual
    try:
        while True:
            df = get_data()
            signal = generate_signals(df)
            if signal:
                place_order(signal)
            time.sleep(60)
    except Exception as e:
        print(f'Error: {e}')
        time.sleep(60)
    finally:
        rendimiento = (capital_actual - capital_inicial) / capital_inicial * 100
        print(f'Capital final: {capital_actual} USD')
        print(f'Número de operaciones: {trades}')
        print(f'Comisiones totales: {comisiones_totales} USD')
        print(f'Rendimiento: {rendimiento:.2f}%')


if __name__ == '__main__':
    run_bot()

Error: single positional indexer is out-of-bounds
Capital final: 10 USD
Número de operaciones: 0
Comisiones totales: 0 USD
Rendimiento: 0.00%
